# KYC-NLI — M3: fine-tune the cross-encoder on a GPU

The local CPU run in `scripts/train_cross_encoder.py` uses a distilled model with
frozen embeddings, because a full fine-tune of anything larger takes hours on four
CPU threads. It answers *"does a learned model beat the rule engine, and where"*.
It does not answer *"how good can this get"*.

This notebook is that second run. On a free Colab T4 it takes roughly 10 minutes.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Everything below
assumes a GPU is present and the first cell will tell you if one is not.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again."
)
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Get the code

Replace the URL with your own remote if you have pushed further work.

In [ ]:
!git clone --depth 1 https://github.com/StutiArya2/kyc-nli.git
%cd kyc-nli
!pip install -q -e ".[nli]"

## 2. Build the benchmark

The datasets are gitignored by design — nothing under `data/` is ever committed —
so regenerate them here. They are fully reproducible from the config and seed, so
this produces byte-identical data to the local run.

Note `--noise moderate`: the noise rates come from the M1 calibration against
Tesseract, not from hand-built guesses. See `docs/m1-ocr-calibration-findings.md`.

In [ ]:
!kyc-nli generate --identities 200 --noise moderate
!kyc-nli compare --bootstrap 200

## 3. Fine-tune

Two differences from the CPU run, both because a GPU makes them affordable:

- `--no-freeze-embeddings` — the embedding table is 68% of the parameters. Freezing
  it halves CPU step time; on a GPU there is no reason to.
- a larger base model — `xlm-roberta-base` has far better coverage of Devanagari,
  Tamil, Bengali and Perso-Arabic than distilbert-multilingual, and tokenises them
  into fewer, more meaningful pieces.

If XLM-R runs out of memory, drop `--batch-size` to 16.

In [ ]:
!python scripts/train_cross_encoder.py \
    --base-model xlm-roberta-base \
    --device cuda \
    --no-freeze-embeddings \
    --epochs 4 \
    --batch-size 32 \
    --learning-rate 2e-5 \
    --max-length 192

## 4. Score it against the baselines

The model is scored by the same harness as the rule engine, with no special-casing:
macro-F1 with bootstrap confidence intervals, per-language, per-phenomenon, contrast
consistency, and a **paired** bootstrap on the difference.

Read the paired test, not the interval overlap. Two overlapping intervals can still
be a significant paired difference.

In [ ]:
!kyc-nli compare
!kyc-nli run-baseline cross_encoder

## 5. The result that actually matters

The revised claim is **not** "the model has a higher macro-F1". It is:

> Under OCR corruption, a learned model recovers decisions the rule engine must
> abstain on, reducing human review load at equal or better error rate.

So the number to look at is **review load at a fixed error rate**, and the shape to
look at is the risk–coverage curve. M1 measured the rule engine abstaining on 79%
of decidable cases at heavy noise. That is the work available to recover.

In [ ]:
from kyc_nli.baselines import RuleBaseline
from kyc_nli.data.io import load_instances
from kyc_nli.evaluation import evaluate, paired_bootstrap, risk_coverage
from kyc_nli.languages import repo_root
from kyc_nli.nli.model import CrossEncoderNLI

instances = load_instances(repo_root() / "data/processed/synth_v0.test.jsonl")
model = CrossEncoderNLI(device="cuda")
model_preds = model.predict_all(instances)
rule_preds = [RuleBaseline().predict(i) for i in instances]

m = evaluate("cross_encoder", instances, model_preds, bootstrap=1000)
r = evaluate("rule_engine", instances, rule_preds, bootstrap=1000)
for report in (r, m):
    print(
        f"{report.system:<15} macro-F1 {report.macro_f1:.4f} "
        f"[{report.macro_f1_ci.low:.3f}, {report.macro_f1_ci.high:.3f}]  "
        f"review load {report.review_load:.3f}  "
        f"over-abstention {report.over_abstention_rate:.3f}"
    )

print()
print(paired_bootstrap(instances, model_preds, rule_preds, "cross_encoder", "rule_engine"))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for name, preds in (("rule engine", rule_preds), ("cross-encoder", model_preds)):
    curve = risk_coverage(instances, preds)
    ax.plot([c for c, _ in curve], [e for _, e in curve], marker="o", ms=3, label=name)
ax.set_xlabel("coverage")
ax.set_ylabel("error rate")
ax.set_title("Risk-coverage: lower is better")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

## 6. Save the checkpoint

`results/` is gitignored, so download the model rather than trying to commit it.

In [ ]:
!zip -qr cross_encoder.zip results/models/cross_encoder
from google.colab import files

files.download("cross_encoder.zip")

## What to check before believing any of this

1. **Per-phenomenon, not the headline.** If the model wins only on the temporal
   cases it has learned a rule engine, not inference.
2. **Contrast-group consistency.** A model exploiting surface string similarity
   scores well on macro-F1 and badly here.
3. **NEUTRAL precision.** Overconfidence on NEUTRAL means deciding on insufficient
   evidence. The rule engine's is 0.000; beating it on macro-F1 while being worse
   here is not an improvement for a KYC system.
4. **Per-language.** Urdu is the hard case (M1: Tesseract returns nothing for
   10–35% of Urdu words). A headline carried by English tells you nothing.